In [1]:
# =========================================================
# STEP 1: IMPORT LIBRARIES
# =========================================================
import numpy as np
import pandas as pd

import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import pickle
import os

In [3]:
# =========================================================
# STEP 2: LOAD DATASET
# =========================================================

df = pd.read_csv('/kaggle/input/maildataset/mail_data.csv')

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (5572, 2)


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# =========================================================
# STEP 3: DATA CLEANING
# =========================================================

# Rename columns (standard format)
df = df.rename(columns={'Category': 'label', 'Message': 'text'})

# Handle missing values
df = df.where(pd.notnull(df), '')

# Convert labels: spam = 0, ham = 1
df['label'] = df['label'].map({'spam': 0, 'ham': 1})

print(df['label'].value_counts())

label
1    4825
0     747
Name: count, dtype: int64


In [5]:
# =========================================================
# STEP 4: TEXT PREPROCESSING
# =========================================================

def preprocess_text(text):
    text = text.lower()  # lowercase
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
df['text'] = df['text'].apply(preprocess_text)

df.head()

,label,text
0,1,go until jurong point crazy available only in ...
1,1,ok lar joking wif u oni
2,0,free entry in a wkly comp to win fa cup final ...
3,1,u dun say so early hor u c already then say
4,1,nah i dont think he goes to usf he lives aroun...


In [6]:
# =========================================================
# STEP 5: TRAIN-TEST SPLIT
# =========================================================

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # IMPORTANT FIX
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (4457,)
Test size: (1115,)


In [7]:
# =========================================================
# STEP 6: TF-IDF VECTORIZATION
# =========================================================

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2)   # unigram + bigram
)

X_train_features = vectorizer.fit_transform(X_train)
X_test_features = vectorizer.transform(X_test)

print("Feature shape:", X_train_features.shape)

Feature shape: (4457, 5000)


In [8]:
# =========================================================
# STEP 7: TRAIN MODELS
# =========================================================

# Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_features, y_train)

# Model 2: Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_features, y_train)

MultinomialNB()

In [9]:
# =========================================================
# STEP 8: EVALUATION
# =========================================================

# Logistic Regression
lr_pred = lr_model.predict(X_test_features)
lr_acc = accuracy_score(y_test, lr_pred)

# Naive Bayes
nb_pred = nb_model.predict(X_test_features)
nb_acc = accuracy_score(y_test, nb_pred)

print("Logistic Regression Accuracy:", lr_acc)
print("Naive Bayes Accuracy:", nb_acc)

Logistic Regression Accuracy: 0.9632286995515695
Naive Bayes Accuracy: 0.9659192825112107


In [10]:
# =========================================================
# STEP 9: SELECT BEST MODEL
# =========================================================

if lr_acc > nb_acc:
    model = lr_model
    best_name = "Logistic Regression"
    y_pred = lr_pred
else:
    model = nb_model
    best_name = "Naive Bayes"
    y_pred = nb_pred

print("Best Model:", best_name)

Best Model: Naive Bayes


In [11]:
# =========================================================
# STEP 10: FINAL METRICS
# =========================================================

print("\nFinal Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Final Accuracy: 0.9659192825112107

Confusion Matrix:
 [[112  37]
 [  1 965]]

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.75      0.85       149
           1       0.96      1.00      0.98       966

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [12]:
# =========================================================
# STEP 11: CUSTOM PREDICTION
# =========================================================

def predict_email(text):
    text = preprocess_text(text)
    vector = vectorizer.transform([text])
    pred = model.predict(vector)[0]
    prob = model.predict_proba(vector)[0]

    label = "Spam" if pred == 0 else "Not Spam"
    confidence = round(max(prob) * 100, 2)

    return label, confidence

# Example
msg = "You have won $1000! Click here now!"
print(predict_email(msg))

('Spam', 70.65)


In [14]:
import pickle
import os

os.makedirs("models", exist_ok=True)

pickle.dump(model, open("models/spam_model.pkl", "wb"))
pickle.dump(vectorizer, open("models/vectorizer.pkl", "wb"))

print("✅ Model saved successfully")

✅ Model saved successfully
